### Required Discussion 16.1: Comparing Models

Now that you have seen a variety of models for regression and classification problems, it is good to step back and weigh the pros and cons of these options.  In the case of classification models, there are at least three things to consider:

1. Is the model good at handling imbalanced classes?
2. Does the model train quickly?
3. Does the model yield interpretable results?

Depending on your dataset and goals, the importance of these considerations will vary from project to project.  Your goal is to review our models to this point and discuss the pros and cons of each.  Two example datasets are offered as a way to offer two very different tasks where interpretability of the model may be of differing importance.

### Data and Task

Your goal is to discuss the pros and cons of Logistic Regression, Decision Trees, KNN, and SVM for the tasks below.  Consider at least the three questions above and list any additional considerations you believe are important to determining the "best" model for the task.  Share your response with your peers on the class discussion board.  

**TASK 1**: Predicting Customer Churn

Suppose you are tasked with producing a model to predict customer churn.  Which of your classification models would you use and what are the pros and cons of this model for this task?  Be sure to consider interpretability, imbalnced classes, and the speed of training.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

The data is loaded below.  Note that the handwritten digit data is already split into features and target (`digits`, `labels`). 

In [ ]:
churn = pd.read_csv('data/telecom_churn.csv')
digits, labels = load_digits(return_X_y=True)

In [ ]:
#churn data
churn.head()

In [ ]:
X = churn.drop('Churn', axis=1)
y = churn['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
y.value_counts()

In [ ]:
def scores(model, model_name):
    preds = model.predict(X_test)
    accuracy = accuracy_score(preds, y_test)
    precision = precision_score(preds, y_test)
    recall = recall_score(preds, y_test)
    f1 = f1_score(preds, y_test)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'model_name': model_name
    }

In [ ]:
transformer = make_column_transformer(
    (OneHotEncoder(drop="if_binary"), make_column_selector(dtype_include=np.object_)),
    (StandardScaler(), make_column_selector(dtype_include=np.number)),
)

transformer_no_scale = make_column_transformer(
    (OneHotEncoder(drop="if_binary"), make_column_selector(dtype_include=np.object_)),
)

In [ ]:
scores_list = []

In [ ]:
tree = Pipeline([
    ('transformer', transformer_no_scale),
    ('tree', DecisionTreeClassifier()),
])
tree.fit(X_train, y_train)
scores_list.append(scores(tree, 'DecisionTreeClassifier'))

In [ ]:
lr = Pipeline([
    ('transformer', transformer),
    ('lr', LogisticRegression()),
])
lr.fit(X_train, y_train)
scores_list.append(scores(lr, 'LogisticRegression'))

In [ ]:
svc = Pipeline([
    ('transformer', transformer),
    ('svc', SVC()),
])
svc.fit(X_train, y_train)
scores_list.append(scores(svc, 'SVC'))

In [ ]:
knn = Pipeline([
    ('transformer', transformer),
    ('knn', KNeighborsClassifier()),
])
knn.fit(X_train, y_train)
scores_list.append(scores(knn, 'KNeighborsClassifier'))

In [ ]:
print(scores_list)
scores_df = pd.DataFrame(scores_list)

In [ ]:
fig = go.Figure()
test_bar = go.Bar(x=scores_df["model_name"], y=scores_df["accuracy"], name="Test accuracy")
fig.add_trace(test_bar)

fig.update_layout(
    title_text="SVC has the highest test accuracy",
    xaxis_title="Model",
    yaxis_title="Accuracy",
    height=600,
    width=1000,
    showlegend=True,
)
fig.show()
fig.write_image("images/accuracy.png")

In [ ]:
fig = go.Figure()
test_bar = go.Bar(x=scores_df["model_name"], y=scores_df["precision"], name="Test precision")
fig.add_trace(test_bar)

fig.update_layout(
    title_text="SVC has the highest test precision",
    xaxis_title="Model",
    yaxis_title="Precision",
    height=600,
    width=1000,
    showlegend=True,
)
fig.show()
fig.write_image("images/precision.png")

In [ ]:
fig = go.Figure()
test_bar = go.Bar(x=scores_df["model_name"], y=scores_df["recall"], name="Test recall")
fig.add_trace(test_bar)

fig.update_layout(
    title_text="SVC has the highest test recall",
    xaxis_title="Model",
    yaxis_title="Recall",
    height=600,
    width=1000,
    showlegend=True,
)
fig.show()
fig.write_image("images/recall.png")

In [ ]:
fig = go.Figure()
test_bar = go.Bar(x=scores_df["model_name"], y=scores_df["f1"], name="Test f1")
fig.add_trace(test_bar)

fig.update_layout(
    title_text="SVC has the highest test f1",
    xaxis_title="Model",
    yaxis_title="F1",
    height=600,
    width=1000,
    showlegend=True,
)
fig.show()
fig.write_image("images/f1.png")

"**TASK 2**: Recognizing Handwritten Digits

Suppose you are tasked with training a model to recognize handwritten digits.  Which of your classifier would you use here and why?  Again, be sure to consider the balance of classes, speed of training, and importance of interpretability.



In [ ]:
#example image
plt.imshow(digits[0].reshape(8, 8))
plt.title('This is a handwritten 0.');

In [ ]:
def scores_digits(model, model_name):
    preds = model.predict(X_test)
    accuracy = accuracy_score(preds, y_test)
    return {
        'accuracy': accuracy,
        'model_name': model_name
    }

In [ ]:
X = digits
y = labels
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
scores_list = []

In [ ]:
tree = DecisionTreeClassifier()
tree.fit(X_train, y_train)
scores_list.append(scores_digits(tree, 'DecisionTreeClassifier'))

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
scores_list.append(scores_digits(lr, 'LogisticRegression'))

In [ ]:
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
scores_list.append(scores_digits(knn, 'KNeighborsClassifier'))

In [ ]:
svc = SVC()
svc.fit(X_train, y_train)
scores_list.append(scores_digits(svc, 'SVC'))

In [ ]:
print(scores_list)
scores_df = pd.DataFrame(scores_list)

In [ ]:
fig = go.Figure()
test_bar = go.Bar(x=scores_df["model_name"], y=scores_df["accuracy"], name="Test accuracy")
fig.add_trace(test_bar)

fig.update_layout(
    title_text="SVC has the highest test accuracy",
    xaxis_title="Model",
    yaxis_title="Accuracy",
    height=600,
    width=1000,
    showlegend=True,
)
fig.show()
fig.write_image("images/digit_accuracy.png")
